# Error-State Extended Kalman Filter for Visual-Inertial Odometry

This notebook walks through **one full step** of a 15-state **Error-State
Extended Kalman Filter (ES-EKF)** used for visual-inertial odometry (VIO).
At every step it shows the math, then the runnable numpy code that
produces the same numbers.

## What is an "error-state" EKF, and why use it here?

The standard EKF (see [`ExtendedKalmanFilter.ipynb`](ExtendedKalmanFilter.ipynb))
estimates the **full state** $\mathbf{x}$ and propagates a Gaussian
$\mathcal{N}(\hat{\mathbf{x}}, P)$ through nonlinear motion and measurement
models. That works when the state lives in a vector space.

For VIO the state contains **rotation**, which lives on the manifold
$SO(3)$, not $\mathbb{R}^3$. A Gaussian over rotation matrices doesn't
make sense — rotations don't add. The fix is to split the state in two:

- **Nominal state** $\hat x$ — stored exactly (position, velocity,
  rotation matrix, biases). Propagated by the full nonlinear IMU
  integration. Not Gaussian.
- **Error state** $\delta x \in \mathbb{R}^{15}$ — a small additive
  perturbation around the nominal. Lives in a vector space, so a
  Gaussian over it *does* make sense. This is what the EKF actually
  tracks.

At measurement time we run a standard linear Kalman update on $\delta x$,
then **inject** the correction back into the nominal state — using
$R \leftarrow R \cdot \mathrm{Exp}(\delta\theta)$ for the rotation block,
which keeps it on $SO(3)$.

## Setup

We assume **monocular VO** gives a relative pose $(R_{vo}, t_{vo})$
between two camera frames, and we **force scale** $|t_{vo}| = 1$ (so
$t_{vo}$ is a *pseudo-metric* translation — see the closing note about
why this is risky). We turn that into a pseudo absolute pose measurement
at time $k{+}1$ by composing with the known pose at time $k$. To keep
the example minimal, the pose at $k$ is identity in the world.


## 1. State and ordering

### Nominal (stored)

$$
\hat x = (\hat p,\ \hat v,\ \hat R,\ \hat b_g,\ \hat b_a)
$$

### Error state (for covariance only)

$$
\delta x = (\delta p,\ \delta v,\ \delta\theta,\ \delta b_g,\ \delta b_a) \in \mathbb{R}^{15}
$$

Index ordering used throughout:

$$
[\delta p\, (0{:}3),\ \delta v\, (3{:}6),\ \delta\theta\, (6{:}9),\ \delta b_g\, (9{:}12),\ \delta b_a\, (12{:}15)]
$$


## 2. SO(3) helpers — `hat`, `Exp`, `Log`

The error-state lives in $\mathbb{R}^{15}$, but the rotation block of the
nominal state is on $SO(3)$. We need three Lie-group helpers:

- $[\cdot]_\times$ (`hat`) — turn a 3-vector into its skew-symmetric
  cross-product matrix.
- $\mathrm{Exp}: \mathbb{R}^3 \to SO(3)$ — Rodrigues' formula. Maps a
  rotation vector to a rotation matrix.
- $\mathrm{Log}: SO(3) \to \mathbb{R}^3$ — inverse of $\mathrm{Exp}$.
  Maps a rotation matrix back to a rotation vector.

$$
[\boldsymbol{\omega}]_\times = \begin{bmatrix} 0 & -\omega_z & \omega_y \\ \omega_z & 0 & -\omega_x \\ -\omega_y & \omega_x & 0 \end{bmatrix}
$$

$$
\mathrm{Exp}(\boldsymbol{\phi}) = I + \frac{\sin\|\boldsymbol{\phi}\|}{\|\boldsymbol{\phi}\|}[\boldsymbol{\phi}]_\times + \frac{1 - \cos\|\boldsymbol{\phi}\|}{\|\boldsymbol{\phi}\|^2}[\boldsymbol{\phi}]_\times^2
$$


In [1]:
import numpy as np
np.set_printoptions(suppress=True, precision=6)


def hat(v):
    # Skew-symmetric cross-product matrix [v]_x
    return np.array([[ 0.0, -v[2],  v[1]],
                     [ v[2],  0.0, -v[0]],
                     [-v[1],  v[0],  0.0]])


def Exp(phi):
    # SO(3) exponential map (Rodrigues' formula). R^3 -> SO(3)
    theta = np.linalg.norm(phi)
    if theta < 1e-9:
        return np.eye(3) + hat(phi)            # first-order Taylor
    K = hat(phi / theta)
    return np.eye(3) + np.sin(theta) * K + (1.0 - np.cos(theta)) * (K @ K)


def Log(R):
    # SO(3) log map. SO(3) -> R^3
    cos_theta = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    if theta < 1e-9:
        return 0.5 * np.array([R[2, 1] - R[1, 2],
                               R[0, 2] - R[2, 0],
                               R[1, 0] - R[0, 1]])
    return (theta / (2.0 * np.sin(theta))) * np.array(
        [R[2, 1] - R[1, 2],
         R[0, 2] - R[2, 0],
         R[1, 0] - R[0, 1]]
    )


# Sanity check: Log(Exp(phi)) == phi
phi_test = np.array([0.1, -0.05, 0.2])
assert np.allclose(Log(Exp(phi_test)), phi_test, atol=1e-12)
print("SO(3) helpers OK")


SO(3) helpers OK


## 3. Given values (time k → k+1)

$$
\Delta t = 0.1\,\text{s}, \quad g = \begin{bmatrix} 0 \\ 0 \\ -9.81 \end{bmatrix}
$$

Nominal state at $k$:

$$
\hat p = \begin{bmatrix} 0\\0\\0 \end{bmatrix},\quad \hat v = \begin{bmatrix} 1\\0\\0 \end{bmatrix},\quad \hat R = I
$$

$$
\hat b_g = \begin{bmatrix} 0.020\\ -0.010\\ 0.005 \end{bmatrix}\,\text{rad/s},\qquad \hat b_a = \begin{bmatrix} 0.10\\ -0.05\\ 0.02 \end{bmatrix}\,\text{m/s}^2
$$

IMU readings at this step:

$$
\boldsymbol{\omega}_m = \begin{bmatrix} 0\\0\\1 \end{bmatrix}\,\text{rad/s}, \qquad \mathbf{a}_m = \begin{bmatrix} 0.4\\0\\9.81 \end{bmatrix}\,\text{m/s}^2
$$

Bias-corrected:

$$
\boldsymbol{\omega} = \boldsymbol{\omega}_m - \hat b_g = \begin{bmatrix} -0.020\\ 0.010\\ 0.995 \end{bmatrix},\quad \mathbf{f} = \mathbf{a}_m - \hat b_a = \begin{bmatrix} 0.30\\ 0.05\\ 9.79 \end{bmatrix}
$$


In [2]:
dt = 0.1
g  = np.array([0.0, 0.0, -9.81])

# Nominal state at k
p_hat = np.zeros(3)
v_hat = np.array([1.0, 0.0, 0.0])
R_hat = np.eye(3)
b_g   = np.array([ 0.02, -0.01, 0.005])
b_a   = np.array([ 0.10, -0.05, 0.02])

# IMU readings
omega_m = np.array([0.0, 0.0, 1.0])
a_m     = np.array([0.4, 0.0, 9.81])

# Bias-corrected
omega = omega_m - b_g
f     = a_m - b_a

print("omega (bias-corrected):", omega)
print("f     (bias-corrected):", f)


omega (bias-corrected): [-0.02   0.01   0.995]
f     (bias-corrected): [0.3  0.05 9.79]


## 4. Nominal propagation

Rotation update (right-multiply by the body-frame increment so the result
stays on $SO(3)$):

$$
\hat R^+ = \hat R \cdot \mathrm{Exp}(\boldsymbol{\omega}\,\Delta t)
$$

World-frame acceleration, then velocity and position by Euler integration:

$$
\mathbf{a} = \hat R^+ \mathbf{f} + g
$$

$$
\hat v^+ = \hat v + \mathbf{a}\,\Delta t
$$

$$
\hat p^+ = \hat p + \hat v\,\Delta t + \tfrac{1}{2}\mathbf{a}\,\Delta t^2
$$

Biases are random-walk in the nominal model — they only move during the
Kalman update via cross-covariance:

$$
\hat b_g^+ = \hat b_g, \qquad \hat b_a^+ = \hat b_a
$$


In [3]:
phi          = omega * dt
R_hat_plus   = R_hat @ Exp(phi)
a_world      = R_hat_plus @ f + g
v_hat_plus   = v_hat + a_world * dt
p_hat_plus   = p_hat + v_hat * dt + 0.5 * a_world * dt**2

print("R_hat_plus =\n", R_hat_plus)
print("a (world)  =", a_world)
print("v_hat_plus =", v_hat_plus)
print("p_hat_plus =", p_hat_plus)


R_hat_plus =
 [[ 0.995053 -0.099337  0.000899]
 [ 0.099335  0.995052  0.002046]
 [-0.001098 -0.001947  0.999998]]
a (world)  = [ 0.30235   0.099587 -0.020451]
v_hat_plus = [ 1.030235  0.009959 -0.002045]
p_hat_plus = [ 0.101512  0.000498 -0.000102]


## 5. Camera (monocular VO) gives relative pose

VO output between frames $k$ and $k{+}1$:

- Relative rotation: yaw $0.09$ rad
  $$
  R_{vo} = \mathrm{Exp}([0,0,0.09]^\top)
  $$
- Relative translation direction, scaled to unit length:
  $$
  t_{vo} = \begin{bmatrix} 1\\0\\0 \end{bmatrix}
  $$

Assuming the world pose at $k$ is identity, the pseudo absolute
measurement at $k{+}1$ is just the relative pose itself:

$$
R_{meas} = R_{vo}, \qquad p_{meas} = t_{vo}
$$


In [4]:
R_vo  = Exp(np.array([0.0, 0.0, 0.09]))
t_vo  = np.array([1.0, 0.0, 0.0])

R_meas = R_vo
p_meas = t_vo

print("R_meas =\n", R_meas)
print("p_meas =", p_meas)


R_meas =
 [[ 0.995953 -0.089879  0.      ]
 [ 0.089879  0.995953  0.      ]
 [ 0.        0.        1.      ]]
p_meas = [1. 0. 0.]


## 6. Measurement residual

We measure position and orientation, so the residual stacks a position
error (linear) and a rotation error (via $\mathrm{Log}$):

$$
r_p = p_{meas} - \hat p^+
$$

$$
r_\theta = \mathrm{Log}\bigl(R_{meas} \cdot (\hat R^+)^\top\bigr)
$$

$$
r = \begin{bmatrix} r_p \\ r_\theta \end{bmatrix} \in \mathbb{R}^6
$$


In [5]:
r_p     = p_meas - p_hat_plus
r_theta = Log(R_meas @ R_hat_plus.T)
r       = np.concatenate([r_p, r_theta])

print("r_p     =", r_p)
print("r_theta =", r_theta)
print("r       =", r)


r_p     = [ 0.898488 -0.000498  0.000102]
r_theta = [ 0.002042 -0.000909 -0.0095  ]
r       = [ 0.898488 -0.000498  0.000102  0.002042 -0.000909 -0.0095  ]


## 7. Measurement Jacobian H

Because position depends only on $\delta p$ and orientation only on
$\delta\theta$:

$$
H = \begin{bmatrix} I_3 & 0 & 0 & 0 & 0 \\ 0 & 0 & I_3 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{6\times 15}
$$


In [6]:
H = np.zeros((6, 15))
H[0:3, 0:3] = np.eye(3)
H[3:6, 6:9] = np.eye(3)

print("H =\n", H.astype(int))


H =
 [[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]]


## 8. Error-state dynamics Jacobian F

Discrete-time Euler-linearized error model:

$$
\delta p^+ = \delta p + \delta v\,\Delta t
$$

$$
\delta v^+ = \delta v - \hat R^+ [\mathbf{f}]_\times \delta\theta\,\Delta t - \hat R^+ \delta b_a\,\Delta t
$$

$$
\delta\theta^+ = (I - [\boldsymbol{\omega}]_\times \Delta t)\,\delta\theta - \delta b_g\,\Delta t
$$

$$
\delta b_g^+ = \delta b_g, \qquad \delta b_a^+ = \delta b_a
$$

Non-trivial blocks of $F$:

- $F_{pv} = I\,\Delta t$
- $F_{v\theta} = -\hat R^+ [\mathbf{f}]_\times\,\Delta t$
- $F_{v b_a} = -\hat R^+\,\Delta t$
- $F_{\theta\theta} = I - [\boldsymbol{\omega}]_\times\,\Delta t$
- $F_{\theta b_g} = -I\,\Delta t$


In [7]:
F = np.eye(15)
F[0:3, 3:6]   = np.eye(3) * dt
F[3:6, 6:9]   = -R_hat_plus @ hat(f) * dt
F[3:6, 12:15] = -R_hat_plus * dt
F[6:9, 6:9]   = np.eye(3) - hat(omega) * dt
F[6:9, 9:12]  = -np.eye(3) * dt

print("F[3:6, 6:9]   (v <- theta block) =\n", F[3:6, 6:9])
print("F[6:9, 6:9]   (theta <- theta)   =\n", F[6:9, 6:9])


F[3:6, 6:9]   (v <- theta block) =
 [[ 0.097255  0.97413  -0.007955]
 [-0.974146  0.097187  0.029355]
 [ 0.006906 -0.031075 -0.000053]]
F[6:9, 6:9]   (theta <- theta)   =
 [[ 1.      0.0995 -0.001 ]
 [-0.0995  1.     -0.002 ]
 [ 0.001   0.002   1.    ]]


## 9. Covariance prediction $P^- = F P F^\top + Q$

Initial covariance (with some cross-correlation so the biases can update
from the very first frame):

| Block | Value |
|---|---|
| $P_{pp}$ | $0.01\, I$ |
| $P_{vv}$ | $0.04\, I$ |
| $P_{\theta\theta}$ | $0.0025\, I$ |
| $P_{b_g b_g}$ | $10^{-4}\, I$ |
| $P_{b_a b_a}$ | $10^{-2}\, I$ |
| $P_{\theta b_g}$ | $5 \cdot 10^{-5}\, I$ |
| $P_{v b_a}$ | $2 \cdot 10^{-3}\, I$ |

Process noise (simple diagonal Euler discretization):

| Source | $\sigma$ |
|---|---|
| Gyro white | $0.01$ |
| Accel white | $0.10$ |
| Gyro RW | $0.001$ |
| Accel RW | $0.01$ |

Discrete $Q$ blocks: $Q_{\bullet\bullet} = \sigma_\bullet^2 \cdot \Delta t \cdot I$.


In [8]:
P = np.zeros((15, 15))
P[0:3,   0:3]   = 0.01   * np.eye(3)   # P_pp
P[3:6,   3:6]   = 0.04   * np.eye(3)   # P_vv
P[6:9,   6:9]   = 0.0025 * np.eye(3)   # P_thth
P[9:12,  9:12]  = 1e-4   * np.eye(3)   # P_bgbg
P[12:15, 12:15] = 1e-2   * np.eye(3)   # P_baba

# Cross-correlations (symmetric)
P[6:9,  9:12]  = 5e-5 * np.eye(3); P[9:12, 6:9]   = 5e-5 * np.eye(3)
P[3:6,  12:15] = 2e-3 * np.eye(3); P[12:15, 3:6]  = 2e-3 * np.eye(3)

# Q
sigma_g, sigma_a, sigma_bg, sigma_ba = 0.01, 0.1, 0.001, 0.01
Q = np.zeros((15, 15))
Q[6:9,   6:9]   = sigma_g**2  * dt * np.eye(3)
Q[3:6,   3:6]   = sigma_a**2  * dt * np.eye(3)
Q[9:12,  9:12]  = sigma_bg**2 * dt * np.eye(3)
Q[12:15, 12:15] = sigma_ba**2 * dt * np.eye(3)

P_minus = F @ P @ F.T + Q

labels = ['p', 'v', 'theta', 'b_g', 'b_a']
for name, lo, hi in zip(labels, range(0, 15, 3), range(3, 16, 3)):
    print(f"diag(P-)_{name}  = {np.diag(P_minus)[lo:hi]}")


diag(P-)_p  = [0.0104 0.0104 0.0104]
diag(P-)_v  = [0.043098 0.0431   0.040703]
diag(P-)_theta  = [0.002526 0.002526 0.002501]
diag(P-)_b_g  = [0.0001 0.0001 0.0001]
diag(P-)_b_a  = [0.01001 0.01001 0.01001]


## 10. Kalman update — bias corrections fall out for free

Measurement noise: scale on $t_{vo}$ is faked, so we inflate position
variance:

$$
R_m = \mathrm{diag}(0.25,\,0.25,\,0.25,\,\ 0.01,\,0.01,\,0.01)
$$

Then the standard Kalman update on the error state:

$$
S = H P^- H^\top + R_m, \qquad K = P^- H^\top S^{-1}, \qquad \delta\hat x = K\,r
$$

**Why this is the whole point of an ES-EKF:** the corrections to $\delta
b_g$ and $\delta b_a$ are *not* zero, even though $H$ has no bias
columns. They come out of $K$ via the cross-covariances $P_{\theta b_g}$
and $P_{v b_a}$. The camera implicitly tells us about the biases through
the way it disagrees with the IMU-propagated pose.


In [9]:
R_m = np.diag([0.25, 0.25, 0.25, 0.01, 0.01, 0.01])

S = H @ P_minus @ H.T + R_m
K = P_minus @ H.T @ np.linalg.inv(S)
delta_x = K @ r

delta_p     = delta_x[0:3]
delta_v     = delta_x[3:6]
delta_theta = delta_x[6:9]
delta_bg    = delta_x[9:12]
delta_ba    = delta_x[12:15]

print("delta_p     =", delta_p)
print("delta_v     =", delta_v)
print("delta_theta =", delta_theta, "rad")
print()
print("delta_bg    =", delta_bg, "rad/s   <- from cross-covariance only")
print("delta_ba    =", delta_ba, "m/s^2   <- from cross-covariance only")


delta_p     = [ 0.035884 -0.00002   0.000004]
delta_v     = [ 0.013649 -0.000496  0.000009]
delta_theta = [ 0.000411 -0.000183 -0.001901] rad

delta_bg    = [ 0.000007 -0.000002 -0.00003 ] rad/s   <- from cross-covariance only
delta_ba    = [ 0.00069 -0.       0.     ] m/s^2   <- from cross-covariance only


## 11. Injection into nominal — and the Joseph-form covariance update

For the additive parts:

$$
\hat p \leftarrow \hat p^+ + \delta\hat p, \quad \hat v \leftarrow \hat v^+ + \delta\hat v, \quad \hat b_g \leftarrow \hat b_g + \delta\hat b_g, \quad \hat b_a \leftarrow \hat b_a + \delta\hat b_a
$$

For the rotation we use the manifold injection — this is what keeps
$\hat R$ on $SO(3)$:

$$
\hat R \leftarrow \hat R^+ \cdot \mathrm{Exp}(\delta\hat\theta)
$$

Covariance via the **Joseph form**, which stays numerically symmetric and
positive-definite even with floating-point error:

$$
P \leftarrow (I - KH)\,P^-\,(I - KH)^\top + K\,R_m\,K^\top
$$

"Reset" of the error state is conceptual only: by construction the mean
of $\delta x$ is now zero, so we just zero-fill the working vector before
the next step.


In [10]:
p_new = p_hat_plus + delta_p
v_new = v_hat_plus + delta_v
R_new = R_hat_plus @ Exp(delta_theta)
b_g_new = b_g + delta_bg
b_a_new = b_a + delta_ba

# Joseph form
I_KH = np.eye(15) - K @ H
P_new = I_KH @ P_minus @ I_KH.T + K @ R_m @ K.T

# Symmetry check (Joseph form is supposed to preserve it exactly)
asymmetry = np.abs(P_new - P_new.T).max()
print(f"P symmetry residual: {asymmetry:.2e}  (should be ~0)")
print()
print("p_new   =", p_new)
print("v_new   =", v_new)
print("b_g_new =", b_g_new)
print("b_a_new =", b_a_new)
print("R_new on SO(3)? det = ", np.linalg.det(R_new), "  R^T R - I =", np.abs(R_new.T @ R_new - np.eye(3)).max())


P symmetry residual: 6.51e-19  (should be ~0)

p_new   = [ 0.137396  0.000478 -0.000098]
v_new   = [ 1.043884  0.009463 -0.002036]
b_g_new = [ 0.020007 -0.010002  0.00497 ]
b_a_new = [ 0.10069 -0.05     0.02   ]
R_new on SO(3)? det =  1.0   R^T R - I = 2.220446049250313e-16


## Verification — does the code reproduce the markdown numbers?

The original walkthrough has hand-computed numbers like
$\delta\hat b_g \approx [+6.85 \times 10^{-6},\ -2.17 \times 10^{-6},\ -3.04 \times 10^{-5}]$
rad/s. We compare:


In [11]:
expected = {
    "p_hat_plus":   np.array([0.101512, 0.000498, -0.000102]),
    "v_hat_plus":   np.array([1.030235, 0.009959, -0.002045]),
    "r_p":          np.array([0.898488, -0.000498, 0.000102]),
    "r_theta":      np.array([0.002042, -0.000909, -0.009500]),
    "delta_p":      np.array([0.035884, -0.0000199, 0.00000408]),
    "delta_v":      np.array([0.013649, -0.000496, 0.00000914]),
    "delta_theta":  np.array([0.000411, -0.000183, -0.001901]),
    "delta_bg":     np.array([+6.85e-6, -2.17e-6, -3.04e-5]),
    "delta_ba":     np.array([+6.90e-4, -3.82e-7, +7.85e-8]),
}
got = {
    "p_hat_plus":   p_hat_plus,
    "v_hat_plus":   v_hat_plus,
    "r_p":          r_p,
    "r_theta":      r_theta,
    "delta_p":      delta_p,
    "delta_v":      delta_v,
    "delta_theta":  delta_theta,
    "delta_bg":     delta_bg,
    "delta_ba":     delta_ba,
}

print(f"{'name':<14} {'max abs err':<14} status")
for name in expected:
    err = np.abs(got[name] - expected[name]).max()
    ok = "OK" if err < 1e-4 else "CHECK"
    print(f"{name:<14} {err:<14.3e} {ok}")


name           max abs err    status
p_hat_plus     2.557e-07      OK
v_hat_plus     2.613e-07      OK
r_p            2.557e-07      OK
r_theta        3.535e-07      OK
delta_p        3.234e-07      OK
delta_v        3.212e-07      OK
delta_theta    4.818e-07      OK
delta_bg       4.451e-09      OK
delta_ba       8.314e-08      OK


## Closing notes

### Why ES-EKF and not a regular EKF for VIO?

| | Regular EKF | Error-state EKF |
|---|---|---|
| Rotation rep. | Euler angles or quaternion — singular / not Gaussian | Rotation matrix kept exactly; small-angle error $\delta\theta$ is Gaussian |
| Linearization point | Always at the current estimate | Always at $\delta x = 0$ — Jacobians simpler, more stable |
| Bias estimation | Couples awkwardly with nonlinear state | Falls out automatically through cross-covariances (this notebook is one example) |
| Used in | Simple navigation, robotics, low-DoF problems | VIO, INS, MSCKF, ROVIO, OpenVINS, basically all modern VI systems |

### Why "T scaled to 1" is risky with monocular

With a single camera, the translation from VO is only a **direction**
(scale is unobservable without extra info). Setting $|t| = 1$ makes it a
pseudo-metric measurement — we must reflect that with a large position
noise ($R_{m,\text{pos}} = 0.25 I$ above). Otherwise the filter
over-trusts a fabricated scale and the position estimate drifts off the
true trajectory. With a stereo rig, a metric calibration, or fusion
against an absolute reference (GPS, depth, loop closure), this concern
goes away.

A cleaner formulation is to use the translation **direction only** as a
bearing constraint, which avoids pretending scale is known — a good
extension exercise.
